# Digikala Recommendation Prediction — Build Final Delivery

This notebook performs no training. It combines the source package, selected model, and prior-run artifacts, runs a smoke test, creates a manifest, and produces a final delivery ZIP.

Attach three inputs: the Transformer notebook output, the evaluation notebook output, and the private Dataset created from `recommendation_prediction_v1_source.zip`. Kaggle may extract ZIP files automatically; this notebook accepts either the ZIP or its extracted directory. Internet is not required.

In [1]:
from __future__ import annotations

import hashlib
import importlib
import json
import shutil
import sys
import zipfile
from pathlib import Path

SEARCH_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working')
STAGE_DIR = OUTPUT_ROOT / 'recommendation_prediction_v1'
FINAL_ZIP = OUTPUT_ROOT / 'recommendation_prediction_v1_final.zip'
MODEL_RELEASE = 'digikala-rec-xlm-roberta-2pct-v1.0.0'

if not SEARCH_ROOT.exists():
    raise RuntimeError('This packaging notebook is intended to run in Kaggle.')
print('Packaging output:', FINAL_ZIP)

Packaging output: /kaggle/working/recommendation_prediction_v1_final.zip


## Discover and validate inputs

In [2]:
def find_unique_file(filename: str) -> Path:
    matches = sorted(path for path in SEARCH_ROOT.rglob(filename) if path.is_file())
    if not matches:
        raise FileNotFoundError(f'{filename} not found under Kaggle Inputs')
    if len(matches) > 1:
        print(f'Warning: multiple {filename}; using {matches[0]}')
    return matches[0]

def find_model_dir() -> Path:
    matches = sorted(
        path for path in SEARCH_ROOT.rglob('best_transformer_encoder')
        if path.is_dir() and (path / 'config.json').is_file()
    )
    if not matches:
        raise FileNotFoundError('best_transformer_encoder not found')
    path = matches[0]
    has_weights = (path / 'model.safetensors').is_file() or (path / 'pytorch_model.bin').is_file()
    if not has_weights or not (path / 'inference_config.json').is_file():
        raise RuntimeError(f'Incomplete model artifact: {path}')
    return path

source_zip_matches = sorted(SEARCH_ROOT.rglob('recommendation_prediction_v1_source.zip'))
source_dir_matches = sorted(
    path for path in SEARCH_ROOT.rglob('recommendation_prediction_v1')
    if path.is_dir() and (path / 'src' / 'recommendation_prediction').is_dir()
)
if source_zip_matches:
    source_input = source_zip_matches[0]
    source_is_zip = True
elif source_dir_matches:
    source_input = source_dir_matches[0]
    source_is_zip = False
else:
    raise FileNotFoundError('Source ZIP or extracted recommendation_prediction_v1 directory not found')
model_source = find_model_dir()
print('Source package:', source_input, '| zip:', source_is_zip)
print('Model artifact:', model_source)

Source package: /kaggle/input/datasets/maslri/digikala-recommendation-delivery-source/recommendation_prediction_v1 | zip: False
Model artifact: /kaggle/input/notebooks/maslri/digikala-transformer-encoders/best_transformer_encoder


## Assemble the package and collect evidence

In [3]:
if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

if source_is_zip:
    extract_root = OUTPUT_ROOT / '_recommendation_source_extract'
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)
    with zipfile.ZipFile(source_input) as archive:
        archive.extractall(extract_root)
    extracted = [
        path for path in extract_root.rglob('recommendation_prediction_v1')
        if path.is_dir() and (path / 'src' / 'recommendation_prediction').is_dir()
    ]
    if len(extracted) != 1:
        raise RuntimeError(f'Expected one source package root, found {extracted}')
    source_package_dir = extracted[0]
else:
    source_package_dir = source_input
shutil.copytree(source_package_dir, STAGE_DIR)

# Compatibility hotfix for source packages created before Transformers 4.57.6.
# Some saved tokenizer configs already carry fix_mistral_regex; passing it again raises TypeError.
predictor_path = STAGE_DIR / 'src' / 'recommendation_prediction' / 'predictor.py'
predictor_source = predictor_path.read_text(encoding='utf-8')
legacy_argument = '            fix_mistral_regex=False,\n'
if legacy_argument in predictor_source:
    predictor_path.write_text(predictor_source.replace(legacy_argument, ''), encoding='utf-8')
    print('Applied tokenizer compatibility hotfix to the staged predictor.')

model_target = STAGE_DIR / 'model' / 'best_transformer_encoder'
model_target.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(model_source, model_target)

inference_config_path = model_target / 'inference_config.json'
inference_config = json.loads(inference_config_path.read_text(encoding='utf-8'))
inference_config['model_version'] = MODEL_RELEASE
inference_config_path.write_text(json.dumps(inference_config, ensure_ascii=False, indent=2), encoding='utf-8')

training_names = [
    'transformer_run_summary.json',
    'transformer_validation_results.csv',
    'sampled_split_manifest.csv',
]
evaluation_names = [
    'recommendation_test_predictions.csv',
    'recommendation_per_class.csv',
    'recommendation_slice_results.csv',
    'recommendation_confusion_matrix.png',
    'recommendation_failure_cases.csv',
    'recommendation_manual_review_sample.csv',
    'recommendation_latency_results.json',
    'recommendation_integration_contract.json',
    'recommendation_evaluation_summary.json',
    'recommendation_release_card.md',
]

for subdir, names in [('training', training_names), ('evaluation', evaluation_names)]:
    target = STAGE_DIR / 'artifacts' / subdir
    target.mkdir(parents=True, exist_ok=True)
    for name in names:
        shutil.copy2(find_unique_file(name), target / name)

print('Staged files:', sum(1 for path in STAGE_DIR.rglob('*') if path.is_file()))

Staged files: 37


## End-to-end model smoke test

In [4]:
sys.path.insert(0, str(STAGE_DIR / 'src'))
module = importlib.import_module('recommendation_prediction')
predictor = module.RecommendationPredictor(model_target)
examples = [
    {'title': 'excellent', 'body': 'excellent quality; I would buy it again'},
    {'title': 'do not buy', 'body': 'the quality was poor and I returned it'},
    {'title': 'average', 'body': 'reasonable for the price, but I expected more'},
]
smoke_results = predictor.predict_batch(examples, batch_size=3)
for result in smoke_results:
    assert result['label'] in {'recommended', 'not_recommended', 'no_idea'}
    assert abs(sum(result['scores'].values()) - 1.0) < 1e-5
print(json.dumps(smoke_results, ensure_ascii=False, indent=2))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[
  {
    "component": "recommendation_prediction",
    "schema_version": "1.0.0",
    "comment_id": null,
    "product_id": null,
    "label": "recommended",
    "scores": {
      "recommended": 0.9897721409797668,
      "not_recommended": 0.002873363671824336,
      "no_idea": 0.007354481145739555
    },
    "confidence_score": 0.9897721409797668,
    "score_margin": 0.9824176598340273,
    "scores_are_calibrated_probabilities": false,
    "model_version": "digikala-rec-xlm-roberta-2pct-v1.0.0",
    "artifact_sha256": "f343a3bcee07c68beaabece504a9efd1f200661e376f0b5235c75c7c9c394cf4",
    "preprocessing_version": "fa_light_v1",
    "source": "model_prediction",
    "latency_ms": 95.86615666667815
  },
  {
    "component": "recommendation_prediction",
    "schema_version": "1.0.0",
    "comment_id": null,
    "product_id": null,
    "label": "not_recommended",
    "scores": {
      "recommended": 0.0011800117790699005,
      "not_recommended": 0.9846199154853821,
      "no_idea": 0.01

## Final manifest and ZIP

In [5]:
def sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

manifest_lines = []
for path in sorted(path for path in STAGE_DIR.rglob('*') if path.is_file() and path.name != 'MANIFEST.sha256'):
    manifest_lines.append(f'{sha256(path)}  {path.relative_to(STAGE_DIR).as_posix()}')
(STAGE_DIR / 'MANIFEST.sha256').write_text('\n'.join(manifest_lines) + '\n', encoding='utf-8')

with zipfile.ZipFile(FINAL_ZIP, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
    for path in sorted(path for path in STAGE_DIR.rglob('*') if path.is_file()):
        archive.write(path, arcname=(Path(STAGE_DIR.name) / path.relative_to(STAGE_DIR)).as_posix())

summary = {
    'decision': 'PASS',
    'model_release': MODEL_RELEASE,
    'model_artifact_sha256': predictor.artifact_sha256,
    'package_zip': str(FINAL_ZIP),
    'package_size_bytes': FINAL_ZIP.stat().st_size,
    'manifest_entries': len(manifest_lines),
    'smoke_test': 'PASS',
}
(OUTPUT_ROOT / 'final_delivery_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "decision": "PASS",
  "model_release": "digikala-rec-xlm-roberta-2pct-v1.0.0",
  "model_artifact_sha256": "f343a3bcee07c68beaabece504a9efd1f200661e376f0b5235c75c7c9c394cf4",
  "package_zip": "/kaggle/working/recommendation_prediction_v1_final.zip",
  "package_size_bytes": 1145221135,
  "manifest_entries": 40,
  "smoke_test": "PASS"
}


In [6]:
from pathlib import Path
import json
import shutil

working_dir = Path("/kaggle/working")
package_dir = working_dir / "recommendation_prediction_v1"
zip_path = working_dir / "recommendation_prediction_v1_final.zip"
summary_path = working_dir / "final_delivery_summary.json"
temporary_extract_dir = working_dir / "_recommendation_source_extract"

# First confirm that the runtime directory is valid.
required_model_dir = (
    package_dir
    / "model"
    / "best_transformer_encoder"
)

assert package_dir.is_dir(), package_dir
assert required_model_dir.is_dir(), required_model_dir
assert (required_model_dir / "config.json").is_file()
assert (
    (required_model_dir / "model.safetensors").is_file()
    or (required_model_dir / "pytorch_model.bin").is_file()
)

# Remove the redundant ZIP while retaining the runtime directory.
if zip_path.exists():
    zip_path.unlink()
    print("Removed duplicate ZIP:", zip_path)

# The temporary packaging directory is no longer needed.
if temporary_extract_dir.exists():
    shutil.rmtree(temporary_extract_dir)
    print("Removed temporary directory:", temporary_extract_dir)

# Update the summary for direct publication as a Kaggle Dataset.
summary = json.loads(summary_path.read_text(encoding="utf-8"))

package_size = sum(
    path.stat().st_size
    for path in package_dir.rglob("*")
    if path.is_file()
)

summary.update({
    "distribution_mode": "kaggle_dataset_directory",
    "package_zip": None,
    "package_root": str(package_dir),
    "package_size_bytes": package_size,
    "zip_removed_to_avoid_duplicate_storage": True,
})

summary_path.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\nFinal outputs:")
for path in working_dir.iterdir():
    print("-", path.name)

Removed duplicate ZIP: /kaggle/working/recommendation_prediction_v1_final.zip
{
  "decision": "PASS",
  "model_release": "digikala-rec-xlm-roberta-2pct-v1.0.0",
  "model_artifact_sha256": "f343a3bcee07c68beaabece504a9efd1f200661e376f0b5235c75c7c9c394cf4",
  "package_zip": null,
  "package_size_bytes": 1145211895,
  "manifest_entries": 40,
  "smoke_test": "PASS",
  "distribution_mode": "kaggle_dataset_directory",
  "package_root": "/kaggle/working/recommendation_prediction_v1",
  "zip_removed_to_avoid_duplicate_storage": true
}

Final outputs:
- final_delivery_summary.json
- __notebook__.ipynb
- recommendation_prediction_v1
